In [ ]:
import random
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from scipy.stats import pearsonr, spearmanr
from transformers import AutoModel, AutoTokenizer

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "distilbert-base-uncased"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 128 if device == "mps" else 32
max_length = 128
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "max_length": max_length,
    "seed": seed,
})

In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["sentence1"] = df["sentence1"].astype(str)
df["sentence2"] = df["sentence2"].astype(str)
df["label"] = df["label"].astype(np.float32)

print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())

In [ ]:
sentences1 = df["sentence1"].tolist()
sentences2 = df["sentence2"].tolist()
labels = df["label"].to_numpy(dtype=np.float32)

all_sentences = sentences1 + sentences2
sentence_counts = Counter(all_sentences)
unique_sentences = list(dict.fromkeys(all_sentences))
sentence_to_idx = {text: idx for idx, text in enumerate(unique_sentences)}

pair_idx1 = np.fromiter((sentence_to_idx[s] for s in sentences1), dtype=np.int32, count=len(sentences1))
pair_idx2 = np.fromiter((sentence_to_idx[s] for s in sentences2), dtype=np.int32, count=len(sentences2))

total_sentence_occurrences = len(all_sentences)
num_unique_sentences = len(unique_sentences)
cache_hits = total_sentence_occurrences - num_unique_sentences
cache_hit_rate = cache_hits / total_sentence_occurrences if total_sentence_occurrences else 0.0

print({
    "total_sentence_occurrences": total_sentence_occurrences,
    "num_unique_sentences": num_unique_sentences,
    "cache_hits": cache_hits,
    "cache_hit_rate": round(cache_hit_rate, 6),
})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()

encoded_unique = tokenizer(
    unique_sentences,
    padding=True,
    truncation=True,
    max_length=max_length,
    return_tensors="pt",
)

input_ids_all = encoded_unique["input_ids"]
attention_mask_all = encoded_unique["attention_mask"]
token_counts = attention_mask_all.sum(dim=1).cpu().numpy().astype(np.int32)

print({
    "tokenized_unique_shape": tuple(input_ids_all.shape),
    "min_tokens": int(token_counts.min()),
    "median_tokens": float(np.median(token_counts)),
    "mean_tokens": float(np.mean(token_counts)),
    "max_tokens": int(token_counts.max()),
})

In [ ]:
def masked_mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

unique_embeddings_batches = []

with torch.no_grad():
    for start in range(0, num_unique_sentences, batch_size):
        end = min(start + batch_size, num_unique_sentences)
        batch_input_ids = input_ids_all[start:end].to(device)
        batch_attention_mask = attention_mask_all[start:end].to(device)

        outputs = model(input_ids=batch_input_ids, attention_mask=batch_attention_mask)
        pooled = masked_mean_pool(outputs.last_hidden_state, batch_attention_mask)
        pooled = F.normalize(pooled, p=2, dim=1)
        unique_embeddings_batches.append(pooled.cpu())

unique_embeddings = torch.cat(unique_embeddings_batches, dim=0).numpy().astype(np.float32)

print({
    "model_name": model_name,
    "embedding_shape": tuple(unique_embeddings.shape),
    "embedding_dtype": str(unique_embeddings.dtype),
})

In [ ]:
emb1 = unique_embeddings[pair_idx1]
emb2 = unique_embeddings[pair_idx2]

cosine_similarity = np.sum(emb1 * emb2, axis=1)
predicted_score_0_5 = 2.5 * (cosine_similarity + 1.0)
absolute_error = np.abs(predicted_score_0_5 - labels)

sentence1_token_count = token_counts[pair_idx1]
sentence2_token_count = token_counts[pair_idx2]
pair_total_tokens = sentence1_token_count + sentence2_token_count
pair_max_tokens = np.maximum(sentence1_token_count, sentence2_token_count)
pair_min_tokens = np.minimum(sentence1_token_count, sentence2_token_count)

results_df = df.copy()
results_df["sentence1_idx"] = pair_idx1
results_df["sentence2_idx"] = pair_idx2
results_df["sentence1_token_count"] = sentence1_token_count
results_df["sentence2_token_count"] = sentence2_token_count
results_df["pair_total_tokens"] = pair_total_tokens
results_df["pair_max_tokens"] = pair_max_tokens
results_df["pair_min_tokens"] = pair_min_tokens
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["absolute_error"] = absolute_error

print(results_df[[
    "sentence1", "sentence2", "label", "sentence1_token_count", "sentence2_token_count",
    "cosine_similarity", "predicted_score_0_5", "absolute_error"
]].head(10))

In [ ]:
pearson_corr = pearsonr(predicted_score_0_5, labels).statistic
spearman_corr = spearmanr(predicted_score_0_5, labels).statistic
mae = float(np.mean(absolute_error))
rmse = float(np.sqrt(np.mean((predicted_score_0_5 - labels) ** 2)))

worst_absolute_error_pairs = (
    results_df[[
        "sentence1", "sentence2", "label", "predicted_score_0_5", "cosine_similarity",
        "absolute_error", "sentence1_token_count", "sentence2_token_count", "pair_total_tokens"
    ]]
    .sort_values(by=["absolute_error", "pair_total_tokens"], ascending=[False, False])
    .head(10)
    .reset_index(drop=True)
)

token_bins = [0, 16, 32, 48, 64, 96, 128, 10**9]
token_labels = ["1-16", "17-32", "33-48", "49-64", "65-96", "97-128", "129+"]
results_df["pair_total_token_bin"] = pd.cut(
    results_df["pair_total_tokens"],
    bins=token_bins,
    labels=token_labels,
    right=True,
    include_lowest=True,
)
results_df["pair_max_token_bin"] = pd.cut(
    results_df["pair_max_tokens"],
    bins=token_bins,
    labels=token_labels,
    right=True,
    include_lowest=True,
)

token_error_summary_total = (
    results_df.groupby("pair_total_token_bin", observed=False)
    .agg(
        num_pairs=("label", "size"),
        mean_label=("label", "mean"),
        mean_prediction=("predicted_score_0_5", "mean"),
        mean_absolute_error=("absolute_error", "mean"),
        median_absolute_error=("absolute_error", "median"),
        max_absolute_error=("absolute_error", "max"),
    )
    .reset_index()
)

token_error_summary_max = (
    results_df.groupby("pair_max_token_bin", observed=False)
    .agg(
        num_pairs=("label", "size"),
        mean_label=("label", "mean"),
        mean_prediction=("predicted_score_0_5", "mean"),
        mean_absolute_error=("absolute_error", "mean"),
        median_absolute_error=("absolute_error", "median"),
        max_absolute_error=("absolute_error", "max"),
    )
    .reset_index()
)

print({
    "pearson_correlation": round(float(pearson_corr), 6),
    "spearman_correlation": round(float(spearman_corr), 6),
    "mae": round(mae, 6),
    "rmse": round(rmse, 6),
})
print(worst_absolute_error_pairs)
print(token_error_summary_total)
print(token_error_summary_max)

In [ ]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"pooling: masked_mean_last_hidden_state")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {len(df)}")
print(f"total_sentence_occurrences: {total_sentence_occurrences}")
print(f"num_unique_sentences: {num_unique_sentences}")
print(f"cache_hits: {cache_hits}")
print(f"cache_hit_rate: {cache_hit_rate:.6f}")
print(f"max_length: {max_length}")
print(f"mean_unique_sentence_tokens: {float(np.mean(token_counts)):.4f}")
print(f"pearson_correlation: {float(pearson_corr):.6f}")
print(f"spearman_correlation: {float(spearman_corr):.6f}")
print(f"mae: {mae:.6f}")
print(f"rmse: {rmse:.6f}")
print("worst_absolute_error_pairs:")
print(worst_absolute_error_pairs.to_dict(orient="records"))
print("token_error_summary_total_tokens:")
print(token_error_summary_total.to_dict(orient="records"))
print("token_error_summary_max_side_tokens:")
print(token_error_summary_max.to_dict(orient="records"))
print(f"runtime_seconds: {runtime_seconds:.2f}")